In [ ]:
!pip -q install requests urllib3

In [ ]:
import bridge_service
import summariser_module
print("modules import OK")

In [ ]:
# --- DEV RELOAD: fast iteration on bridge_service.py within a live runtime ---
# Use when you've edited the .py files and want changes without a full restart.
# LIMITATION: importlib.reload does NOT reliably pick up dataclass field changes,
# altered module-level constants, or nested-import changes. If behavior looks
# stale after a reload, use Runtime > Restart runtime and re-run from the top.

###Reload the module
import importlib
importlib.reload(bridge_service)
importlib.reload(summariser_module)
###Recreate the bridge
from bridge_service import make_config_from_env, MoltbookMyceliumBridge
cfg = make_config_from_env()
bridge = MoltbookMyceliumBridge(cfg)
print(cfg)

In [ ]:
"""
Bridge Service Execution Controller (orchestration only).

Logic lives in bridge_service.py / summariser_module.py — edit those .py files,
not this notebook. This cell only loads config and drives one run.
"""
import json
import os
import urllib3
from bridge_service import make_config_from_env, MoltbookMyceliumBridge
from google.colab import userdata

urllib3.disable_warnings()

# ---------- 0. Secrets (from Colab Secrets panel) ----------
_SECRET_ENV_MAP = {
    "LITELLM_API_KEY":               "LITELLM_API_KEY",
    "MOLTBOOK_API_KEY":              "MOLTBOOK_API_KEY",
    "MOLTBOOK_API_KEY_FINANCEGUARD": "MOLTBOOK_API_KEY_FINANCEGUARD",
    "MOLTBOOK_API_KEY_CAREERMAX":    "MOLTBOOK_API_KEY_CAREERMAX",
    "MOLTBOOK_API_KEY_STUDENTLIFE":  "MOLTBOOK_API_KEY_STUDENTLIFE",
    "MOLTBOOK_API_KEY_RISKANALYST":  "MOLTBOOK_API_KEY_RISKANALYST",
}

missing = []
for secret_name, env_var in _SECRET_ENV_MAP.items():
    try:
        os.environ[env_var] = userdata.get(secret_name)
    except userdata.SecretNotFoundError:
        missing.append(secret_name)
    except userdata.NotebookAccessError:
        raise RuntimeError(
            f"Notebook not granted access to secret '{secret_name}'. "
            "Toggle 'Notebook access' ON in the Colab Secrets panel."
        )

if missing:
    raise RuntimeError(f"Add these secrets in the Colab Secrets panel: {missing}")

# ---------- 1. Core Parameters ----------
os.environ['MOLTBOOK_BASE_URL'] = 'https://www.moltbook.com/api/v1'
os.environ['MOLTBOOK_SUBMOLT'] = 'secure-dev-logic'
os.environ['MOLTBOOK_SUBMOLT_DISPLAY'] = 'Private Agent Logic Exchange'
os.environ['MOLTBOOK_SUBMOLT_DESCRIPTION'] = 'A restricted room for back-end multi-agent synchronization.'
os.environ['MYCELIUM_BASE_URL'] = '<MYCELIUM_SECURE_TUNNEL_URL_HERE>'
os.environ["MYCELIUM_ROOM"] = "moltbook-mycelium-demo"
os.environ['MYCELIUM_SENDER'] = 'amlan'
os.environ["BRIDGE_STATE_PATH"] = "bridge_state.json"

# ---------- 2. LiteLLM Parameters ----------
os.environ["LITELLM_MODEL"] = "azure/gpt-4o-mini"
os.environ["LITELLM_URL"] = "https://litellm.prod.outshift.ai/v1/chat/completions"


# ---------- 3. Per-experiment inputs ----------
# Scenario facts are injected into each agent lens by bridge_service; override per run.
os.environ.setdefault(
    "SCENARIO_FACTS",
    "ASU has a $25,000/year scholarship and NYU has no aid.",
)
# Target post to (re)run. Optional title check runs only if EXPECTED_TOPIC_TITLE is set.
os.environ.setdefault("TARGET_POST_ID", "53fff94b-6542-4b91-bdcc-aabf9c9d237a")
os.environ.setdefault("EXPECTED_TOPIC_TITLE", "")

# ---------- 4. Load config + build bridge ----------
cfg = make_config_from_env()
bridge = MoltbookMyceliumBridge(cfg)
print("###################################################")
print("parameters load and configuration done")
print("###################################################")
print(cfg)

# ---------- 5. Bootstrapping environment checks ----------
submolt = bridge.ensure_submolt()
print("###################################################")
print("environment checks done")
print("###################################################")
print("SUBMOLT READY PROFILE:", submolt)

# ---------- 6. Smoke test Moltbook posts and Mycelium messages ----------
print(bridge.moltbook.list_submolt_posts(cfg.submolt_name, limit=5))
print(bridge.mycelium.get_messages(cfg.mycelium_room, limit=5))

posts = bridge.moltbook.list_submolt_posts(cfg.submolt_name, limit=50)
print("###################################################")
print("list submolt posts done")
print("###################################################")
for p in posts:
    print(p.get("id"), "|", p.get("title") or p.get("name"))

# ---------- 7. Optional post title verification ----------
expected_title = os.environ.get("EXPECTED_TOPIC_TITLE", "").strip()
if expected_title:
    print("###################################################")
    print("post name discovery done/ title verification")
    print("###################################################")
    for p in posts:
        title = (p.get("title") or p.get("name") or "").strip()
        if title.lower() == expected_title.lower():
            print("FOUND:", p)
            break
    else:
        raise ValueError(f"Expected topic title not found: {expected_title!r}")
else:
    print("NOTE: EXPECTED_TOPIC_TITLE not set — skipping title verification.")


# ---------- 8. Trigger orchestration loop ----------
target_post_id = os.environ["TARGET_POST_ID"]
print("###################################################")
print(f"Triggering orchestration loop for ID: {target_post_id}...")
print("###################################################")
result = bridge.rerun_topic_by_id(
    target_post_id,
    post_kickoff_comment=True,
    post_final_comment=True,
    force=True,
)
print("\n--- LOOP PROCESS COMPLETED ---")
print(json.dumps(result, indent=2))